# ADS-B の生の1通を手で分解する

kikicom の受信機(月寒、RTL-SDR + readsb)が受けた**デコード前のフレーム**を、ビット単位で自分の手で分解するノート。
readsb が普段やっていることを一つずつ再現し、最後に readsb の解釈と答え合わせをする。

**扱うこと**

1. Beast 形式(readsb の生フレーム出力)を読む
2. 112ビットの中身:DF・CA・ICAO アドレス・ME・パリティ
3. CRC-24:誤りの検出と、1ビット誤りの訂正
4. 識別メッセージ:コールサインと機体区分
5. 位置メッセージ:高度と CPR(偶数・奇数の2通で位置が決まる)
6. 速度メッセージ:東西・南北成分から対地速度と進行方向
7. Mode S の応答(DF4/5/11/20):ICAO アドレスが「パリティに溶かし込まれている」仕組み
8. 衝突をどう避けているか:送信間隔のランダムなゆらぎを、受信時刻から見る

**データ**:`~/kikicom-data/raw/beast-*.bin`(RPi の readsb `--net-bo-port 30005` を `nc` で数分記録したもの)と、
答え合わせ用の `~/kikicom-data/adsb-log/*.jsonl`。どちらも生データなのでリポジトリには入れない(このノートもコミット時は出力を消す)。

**記録のしかた**(slate から):
```bash
ssh m329.local 'timeout 300 nc 127.0.0.1 30005' > ~/kikicom-data/raw/beast-$(date +%Y%m%d-%H%M).bin
```

In [ ]:
import glob, os, json, collections, math, datetime
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['Hiragino Sans', 'Hiragino Maru Gothic Pro', 'sans-serif']  # macOS の日本語フォント

RAW = os.path.expanduser('~/kikicom-data/raw')
BEAST = sorted(glob.glob(os.path.join(RAW, 'beast-*.bin')))[-1]
print('使うファイル:', BEAST, os.path.getsize(BEAST), 'bytes')

## 1. Beast 形式を読む

readsb が外に出す「生フレーム」の形式。1フレームはこう並ぶ:

| バイト | 中身 |
|---|---|
| `0x1a` | 区切り |
| 1 | 種類:`'1'`=Mode A/C(2バイト)、`'2'`=Mode S 短(7バイト=56ビット)、`'3'`=Mode S 長(14バイト=112ビット) |
| 6 | 受信時刻:**12MHz のカウンタ**(RTL-SDR のサンプル数から作った受信機ローカルの時計) |
| 1 | 信号の強さ |
| 2/7/14 | フレーム本体(これが空を飛んできたビット列) |

本体に `0x1a` が現れると `0x1a 0x1a` と2回書く(エスケープ)。

In [ ]:
LEN = {0x31: 2, 0x32: 7, 0x33: 14}

def read_beast(path):
    d = open(path, 'rb').read()
    i, out = 0, []
    while i < len(d):
        if d[i] != 0x1a or i + 1 >= len(d) or d[i + 1] not in LEN:
            i += 1
            continue
        kind = d[i + 1]
        need = 6 + 1 + LEN[kind]
        j, buf = i + 2, bytearray()
        while len(buf) < need and j < len(d):
            if d[j] == 0x1a:
                if j + 1 < len(d) and d[j + 1] == 0x1a:
                    buf.append(0x1a); j += 2; continue
                break
            buf.append(d[j]); j += 1
        if len(buf) == need:
            out.append({'ts12': int.from_bytes(buf[:6], 'big'),   # 12MHz カウンタ
                        'sig': buf[6],
                        'msg': bytes(buf[7:])})
        i = j
    return out

frames = [f for f in read_beast(BEAST) if len(f['msg']) in (7, 14)]
for f in frames:
    f['df'] = f['msg'][0] >> 3          # 先頭5ビット
print(len(frames), 'フレーム')
print('DF ごとの数:', dict(sorted(collections.Counter(f['df'] for f in frames).items())))

**DF(Downlink Format)** は先頭5ビットで、フレームの種類を表す:

| DF | 長さ | 何か |
|---|---|---|
| 4 | 56 | 地上レーダーの問いかけへの応答:**高度** |
| 5 | 56 | 地上レーダーの問いかけへの応答:**スコーク** |
| 11 | 56 | 「全機呼び出し」への応答(機体が自分の存在を知らせる) |
| **17** | 112 | **ADS-B(Extended Squitter)**:問いかけ無しで自発的に送る。今回の主役 |
| 20/21 | 112 | 応答+データ(Comm-B:対気速度・機首方位など) |

DF4/5/20/21 は**地上のレーダー(千歳・札幌の管制レーダー)が問いかけた結果の返事**を、横から聞いている。

## 2. 112ビットの中身(DF17)

```
| DF(5) | CA(3) | ICAO アドレス(24) |        ME:本文(56)        | PI:パリティ(24) |
```

In [ ]:
def bits_of(msg):
    return ''.join(f'{b:08b}' for b in msg)

def field(bits, start, length):
    return int(bits[start:start + length], 2)

df17 = [f for f in frames if f['df'] == 17]
for f in df17:
    b = bits_of(f['msg'])
    f['icao'] = f'{field(b, 8, 24):06x}'
    f['tc'] = field(b, 32, 5)            # ME の先頭5ビット = Type Code

by_icao = collections.Counter(f['icao'] for f in df17)
print('DF17 を送ってきた機体:', by_icao.most_common())

# 一番たくさん受かった機体の、最初の1通を取り上げる
ICAO = by_icao.most_common(1)[0][0]
one = next(f for f in df17 if f['icao'] == ICAO)
b = bits_of(one['msg'])
print('\n16進 :', one['msg'].hex())
print('2進  :', b)
print(f"DF   = {field(b, 0, 5)}  CA = {field(b, 5, 3)}  ICAO = {field(b, 8, 24):06x}")
print(f"ME   = {b[32:88]}  (Type Code = {field(b, 32, 5)})")
print(f"PI   = {field(b, 88, 24):06x}")

**Type Code(TC)** が ME の読み方を決める:1〜4=識別(コールサイン)、9〜18=空中の位置(気圧高度)、19=速度、20〜22=空中の位置(GNSS 高度)、28=緊急/優先、29=目標状態(オートパイロットの設定)、31=運用状態(ADS-B のバージョンや精度)。

この短い時間に届いた DF17 の TC の内訳:

In [ ]:
TC_NAME = {**{t: '識別' for t in range(1, 5)}, **{t: '位置(気圧高度)' for t in range(9, 19)},
           19: '速度', **{t: '位置(GNSS高度)' for t in range(20, 23)},
           28: '緊急/優先', 29: '目標状態', 31: '運用状態'}
for tc, n in sorted(collections.Counter(f['tc'] for f in df17).items()):
    print(f'TC {tc:2d} {TC_NAME.get(tc, "?"):12s} {n:4d} 通')

## 3. CRC-24:本物かどうかをどう決めるか

最後の24ビット(PI)は、前の88ビットから計算した **CRC-24** 。生成多項式は `0x1FFF409`。
**112ビット全体を多項式で割った余りが 0 なら、途中で化けていない**と判断する。

In [ ]:
GEN = 0x1FFF409

def crc_remainder(bits):
    # ビット列全体(データ+パリティ)を GEN で割った余り(24ビット)
    v, n = int(bits, 2), len(bits)
    for i in range(n - 24):
        if (v >> (n - 1 - i)) & 1:
            v ^= GEN << (n - 25 - i)
    return v & 0xFFFFFF

print('本物の1通の余り:', f'{crc_remainder(b):06x}')

# わざと1ビット壊してみる
k = 50
broken = b[:k] + ('1' if b[k] == '0' else '0') + b[k + 1:]
print(f'{k}ビット目を反転した余り:', f'{crc_remainder(broken):06x}  ← 0 でないので「壊れている」とわかる')

### 1ビットの誤りなら直せる

余り(シンドローム)は「どのビットが壊れたか」だけで決まる。112通りの1ビット誤りについて余りを事前に表にしておけば、
壊れたフレームの余りを引くだけで**壊れた位置がわかり、直せる**。readsb はこれ(と2ビット版)で弱い信号を救っている。

In [ ]:
zero = '0' * 112
syndrome = {}
for i in range(5, 112):                     # 先頭5ビット(DF)は訂正対象にしない
    e = zero[:i] + '1' + zero[i + 1:]
    syndrome[crc_remainder(e)] = i

s = crc_remainder(broken)
pos = syndrome.get(s)
print('壊れたフレームの余り', f'{s:06x}', '→ 表を引くと', pos, 'ビット目')
fixed = broken[:pos] + ('1' if broken[pos] == '0' else '0') + broken[pos + 1:]
print('直したフレームは元と一致:', fixed == b)

## 4. 識別メッセージ(TC 1〜4):コールサインと機体区分

ME の中で、TC(5)・区分(3)のあとに **6ビット × 8文字**。文字表は独特で、ASCII ではない。
TC と区分の組で機体区分(A1〜A7 など)が決まる。`A0` は「情報なし」。

In [ ]:
CHARSET = '#ABCDEFGHIJKLMNOPQRSTUVWXYZ##### ###############0123456789######'

def decode_ident(bits):
    tc, ca = field(bits, 32, 5), field(bits, 37, 3)
    cs = ''.join(CHARSET[field(bits, 40 + 6 * i, 6)] for i in range(8))
    category = 'ABCD'[4 - tc] + str(ca)     # TC4=A, TC3=B, TC2=C, TC1=D
    return cs.strip('#').strip(), category

idents = {}
for f in df17:
    if 1 <= f['tc'] <= 4:
        idents[f['icao']] = decode_ident(bits_of(f['msg']))
for icao, (cs, cat) in idents.items():
    print(icao, repr(cs), cat)

## 5. 位置メッセージ(TC 9〜18):高度と CPR

```
| TC(5) | 監視状態(2) | 単一アンテナ(1) | 高度(12) | 時刻(1) | F:偶奇(1) | 緯度CPR(17) | 経度CPR(17) |
```

**高度**:12ビットの中の Q ビット(8番目)が 1 なら、残り11ビット × 25ft − 1000ft。

**CPR(Compact Position Reporting)**:緯度経度をそれぞれ17ビットに圧縮する。そのままでは「ある区画の中のどこか」しかわからないので、
**偶数(F=0)と奇数(F=1)の2種類の区画割りで送り、両方をそろえて初めて地球上の1点に決まる**。
readsb のログにあった「global CPR」は、この組を解いた回数。

In [ ]:
def decode_alt(bits):
    a = field(bits, 40, 12)
    q = (a >> 4) & 1
    if not q:
        return None                           # Q=0 は古いギルハム符号(ここでは扱わない)
    n = ((a >> 5) << 4) | (a & 0xF)           # Q ビットを抜いた11ビット
    return n * 25 - 1000

def NL(lat):
    # 経度方向の区画数(緯度によって変わる)
    if abs(lat) >= 87:
        return 1
    return math.floor(2 * math.pi / math.acos(1 - (1 - math.cos(math.pi / 30)) / math.cos(math.radians(lat)) ** 2))

def cpr_global(even, odd):
    # even/odd: (lat_cpr, lon_cpr, t)。新しい方の位置を返す
    lat0, lon0 = even[0] / 131072, even[1] / 131072
    lat1, lon1 = odd[0] / 131072, odd[1] / 131072
    j = math.floor(59 * lat0 - 60 * lat1 + 0.5)
    rlat0 = 6 * ((j % 60) + lat0)
    rlat1 = 360 / 59 * ((j % 59) + lat1)
    rlat0 -= 360 if rlat0 >= 270 else 0
    rlat1 -= 360 if rlat1 >= 270 else 0
    if NL(rlat0) != NL(rlat1):
        return None                           # 区画の境をまたいだ:次の組を待つ
    if even[2] >= odd[2]:
        nl = NL(rlat0); ni = max(nl, 1)
        m = math.floor(lon0 * (nl - 1) - lon1 * nl + 0.5)
        lon = 360 / ni * ((m % ni) + lon0); lat = rlat0
    else:
        nl = NL(rlat1); ni = max(nl - 1, 1)
        m = math.floor(lon0 * (nl - 1) - lon1 * nl + 0.5)
        lon = 360 / ni * ((m % ni) + lon1); lat = rlat1
    lon -= 360 if lon >= 180 else 0
    return lat, lon

pos_frames = [f for f in df17 if 9 <= f['tc'] <= 18]
last = {}
decoded = []
for f in pos_frames:
    bb = bits_of(f['msg'])
    flag = field(bb, 53, 1)
    cpr = (field(bb, 54, 17), field(bb, 71, 17), f['ts12'])
    last.setdefault(f['icao'], {})[flag] = cpr
    pair = last[f['icao']]
    if 0 in pair and 1 in pair and abs(pair[0][2] - pair[1][2]) < 10 * 12_000_000:
        p = cpr_global(pair[0], pair[1])
        if p:
            decoded.append((f['icao'], f['ts12'], decode_alt(bb), p))

print(len(pos_frames), '通の位置メッセージから', len(decoded), '点を復元')
for icao, ts, alt, (lat, lon) in decoded[:8]:
    print(f'{icao} {idents.get(icao, ("?",))[0]:8s} 高度 {alt} ft  {lat:.5f}, {lon:.5f}')

### readsb の解釈と答え合わせ

同じ機体・同じ時間帯の readsb のログ(`adsb-log`)と、手で解いた位置を比べる。
Beast の時刻は受信機ローカルの 12MHz カウンタなので壁時計の時刻とは直接は合わない。ここでは**同じ機体の、手で解いた位置に最も近い readsb の位置**との差を見る。

In [ ]:
logs = []
for p in sorted(glob.glob(os.path.expanduser('~/kikicom-data/adsb-log/*.jsonl'))):
    with open(p) as fh:
        for line in fh:
            try:
                r = json.loads(line)
            except ValueError:
                continue
            if 'lat' in r:
                logs.append(r)
by_hex = collections.defaultdict(list)
for r in logs:
    by_hex[r['hex']].append(r)

def km(a, b):
    return 6371 * math.acos(min(1, math.sin(math.radians(a[0])) * math.sin(math.radians(b[0])) +
                               math.cos(math.radians(a[0])) * math.cos(math.radians(b[0])) *
                               math.cos(math.radians(a[1] - b[1]))))

for icao, ts, alt, (lat, lon) in decoded[:8]:
    cand = by_hex.get(icao)
    if not cand:
        print(icao, 'readsb のログに見当たらない'); continue
    r = min(cand, key=lambda r: km((lat, lon), (r['lat'], r['lon'])))
    print(f"{icao} 手: {lat:.5f},{lon:.5f} {alt}ft | readsb: {r['lat']:.5f},{r['lon']:.5f} {r.get('alt_baro')}ft"
          f" | 差 {km((lat, lon), (r['lat'], r['lon'])) * 1000:.0f} m")

## 6. 速度メッセージ(TC 19)

サブタイプ1(対地速度)では、**東西成分と南北成分**がそれぞれ符号1ビット+大きさ10ビットで送られる。
対地速度 = √(東西² + 南北²)、進行方向 = atan2(東西, 南北)。昇降率は 64ft/分 単位。

In [ ]:
def decode_velocity(bits):
    st = field(bits, 37, 3)
    if st not in (1, 2):
        return None                          # 3,4 は対気速度+機首方位(ここでは扱わない)
    mult = 1 if st == 1 else 4               # 2 は超音速用
    s_ew, v_ew = field(bits, 45, 1), field(bits, 46, 10) - 1
    s_ns, v_ns = field(bits, 56, 1), field(bits, 57, 10) - 1
    vx = (-1 if s_ew else 1) * v_ew * mult   # 東向きが正
    vy = (-1 if s_ns else 1) * v_ns * mult   # 北向きが正
    gs = math.hypot(vx, vy)
    trk = math.degrees(math.atan2(vx, vy)) % 360
    s_vr, vr = field(bits, 69, 1), field(bits, 70, 9)
    rate = None if vr == 0 else (-1 if s_vr else 1) * (vr - 1) * 64
    return round(gs, 1), round(trk, 1), rate

for f in [f for f in df17 if f['tc'] == 19][:8]:
    v = decode_velocity(bits_of(f['msg']))
    if v:
        print(f"{f['icao']} {idents.get(f['icao'], ('?',))[0]:8s} 対地速度 {v[0]} kt  進行方向 {v[1]}°  昇降率 {v[2]} ft/分")

## 7. Mode S の応答:ICAO アドレスは「パリティに溶かし込まれている」

DF4/5/20/21 の56(112)ビットには、ICAO アドレスの欄が**無い**。代わりに、最後の24ビットが
**「CRC」と「ICAO アドレス」の排他的論理和**(Address/Parity)になっている。

だから **フレーム全体の CRC 余りを計算すると、それがそのまま ICAO アドレス**になる。
地上レーダーは「誰に問いかけたか」を知っているので、余りが相手のアドレスと一致すれば正しく届いたと判断できる。
横で聞いている私たちは、余りを計算して「誰の応答か」を知る(本物かどうかは、既に DF17 で知っている機体と一致するかで確かめる)。

In [ ]:
known = {f['icao'] for f in df17}

def decode_ac13(bits):
    # DF4/20 の高度(13ビット):M=0, Q=1 のときの25ft 単位
    ac = field(bits, 19, 13)
    m, q = (ac >> 6) & 1, (ac >> 4) & 1
    if m or not q:
        return None
    n = ((ac >> 7) << 5) | (((ac >> 5) & 1) << 4) | (ac & 0xF)
    return n * 25 - 1000

def decode_squawk(bits):
    # DF5/21 の ID(13ビット):C1 A1 C2 A2 C4 A4 X B1 D1 B2 D2 B4 D4 の並び
    id13 = field(bits, 19, 13)
    g = lambda k: (id13 >> (12 - k)) & 1
    a = g(5) * 4 + g(3) * 2 + g(1)
    b_ = g(11) * 4 + g(9) * 2 + g(7)
    c = g(4) * 4 + g(2) * 2 + g(0)
    d = g(12) * 4 + g(10) * 2 + g(8)
    return f'{a}{b_}{c}{d}'

rows = []
for f in frames:
    if f['df'] in (4, 5, 20, 21):
        bb = bits_of(f['msg'])
        addr = f'{crc_remainder(bb):06x}'
        val = decode_ac13(bb) if f['df'] in (4, 20) else decode_squawk(bb)
        rows.append((f['df'], addr, addr in known, val))
hit = sum(r[2] for r in rows)
print(f'DF4/5/20/21 の {len(rows)} 通のうち、余りが既知の DF17 機体のアドレスと一致したもの {hit} 通')
print()
for (df, addr, ok, val), n in collections.Counter(rows).most_common():
    print(f"DF{df:2d} 余り(=ICAO) {addr} {'既知' if ok else '  - '}  {'高度' if df in (4, 20) else 'スコーク'} {val}  × {n} 通")

余りが既知のアドレスと一致しないものは、(a) まだ DF17 を受けていない機体、(b) ADS-B を送っていない機体(Mode S のみ)、
(c) 途中で化けた応答、のどれか。**(b) が、ヘリや自衛隊機を「位置は無いが、いた」と記録できる理由**。
DF11(全機呼び出しへの応答)は余りの下位7ビットに問いかけ元の識別子が入るので、別の扱いになる。

## 8. 衝突をどう避けているか:送信間隔のゆらぎ

ADS-B には送信の順番を決める調整役がいない(同じ 1090MHz を、周りの全機体と地上レーダーへの応答が共有している)。
仕様では、位置メッセージは **0.4〜0.6秒の間でランダムな間隔**で送る。毎回ずらすことで、
二機がたまたま同時に送って重なっても、次の回まで重なり続けることはない(確率的な衝突回避)。

同じ機体の位置メッセージの受信間隔を、Beast の 12MHz 時計で測る。取りこぼし(受信できなかった回)があると、
間隔は 0.5秒の2倍・3倍…に伸びる。**受信率が低いと、この分布は長い間隔の側にずれ、0.4〜0.6秒の山は見えなくなる**。

In [ ]:
gaps = []
sent = 0
for icao in by_icao:
    ts = sorted(f['ts12'] for f in pos_frames if f['icao'] == icao)
    if len(ts) > 1:
        sent += (ts[-1] - ts[0]) / 12e6 / 0.5      # 0.5秒ごとに送っているはずの数
    gaps += [(b_ - a) / 12e6 for a, b_ in zip(ts, ts[1:]) if (b_ - a) / 12e6 < 15]
gaps = np.array(gaps)
print(f'受信した位置メッセージ {len(pos_frames)} 通 / 送られたはずの数 約 {sent:.0f} 通'
      f'(受信できた割合 約 {len(pos_frames) / max(sent, 1) * 100:.0f}%)')
print(len(gaps), '個の間隔。0.3〜0.7秒に入るもの:', int(((gaps > 0.3) & (gaps < 0.7)).sum()))
plt.figure(figsize=(8, 3))
plt.hist(gaps, bins=np.arange(0, 6.01, 0.1), color='#3498db')
for k in range(1, 11):
    plt.axvspan(0.4 * k, 0.6 * k, color='orange', alpha=0.08)
plt.xlim(0, 6); plt.xlabel('同じ機体の位置メッセージの受信間隔(秒)'); plt.ylabel('回数')
plt.title('橙の帯:仕様上の送信間隔 0.4–0.6秒 と、その整数倍(取りこぼし)')
plt.tight_layout(); plt.show()

2026-09-21 21:25〜21:30 の記録では、受けられた位置メッセージは送られたはずの数の約16%で、0.4〜0.6秒の山は見えなかった。
台風の夜で機体が1機しかおらず、しかも高度33,000ft を遠くで通過していたため。
**ゆらぎそのものを見るには、近くを通る機体(丘珠・新千歳の発着)が多い昼間に記録を取り直す**。
同じ機体の受信率が高ければ、0.4〜0.6秒の幅に広がった山が現れるはず。

**次に確かめること**:この記録では受信間隔が約1・2・3秒に固まり、0.5秒の奇数倍(1.5秒など)がほとんど無かった。
位置メッセージは偶数(F=0)と奇数(F=1)を交互に送るので、**片方だけが受かりやすい**(あるいは readsb が
片方しか Beast に出していない)可能性がある。偶奇ごとに分けて間隔を数えてみる。

## まとめと次の一歩

- 1通の DF17 は「誰(ICAO)」「何(TC)」「中身(ME)」「本物の証明(CRC)」でできている
- 位置は1通では決まらず、偶数・奇数の2通がそろって初めて決まる(CPR)
- Mode S の応答はアドレスをパリティに溶かし込んでいて、余りを計算すると送り主がわかる
- 衝突は「調整」ではなく「ランダムにずらす」ことで確率的に避けている

**次のノート(予定)**:電波の波形そのもの。RTL-SDR の生 IQ から、8μs のプリアンブル(0・1・3.5・4.5μs のパルス)と
1μs ごとのビット(前半が高ければ1、後半が高ければ0)を目で見る。2026-09-21 夜の試し撮りでは、生 IQ の値が
8ビットのうち ±3 程度しか振れておらず(ゲイン49.6dB でも A/D 変換器への入力が小さい)、強い機体が近くにいないと
波形からは読み取れなかった。丘珠に降りる機体が頭上を通るときに取り直す。